In [ ]:
# CELL 1: CRITICAL FIX - Copy this exactly
import subprocess, sys

print("Installing exact compatible versions...")

# Remove conflicts
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", 
                "tensorflow", "tf-keras", "jax", "jaxlib", 
                "numpy", "scipy", "scikit-learn"], 
               capture_output=True)

# Install EXACT working versions
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy==1.23.5",      # Compatible with scipy 1.10
                "scipy==1.10.1",       # No ufunc errors
                "scikit-learn==1.3.2", # Works with numpy 1.23
                "transformers==4.30.2",
                "albumentations==1.3.1",
                "opencv-python-headless==4.8.1.78"],
               capture_output=True)

print("✓ Installation complete!")
print("⚠️  NOW: Click 'Session → Restart Session' then run from Cell 2")


In [ ]:
# ==================== CELL 2: IMPORTS (RUN AFTER KERNEL RESTART) ====================
# Suppress warnings first
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Add this line

import warnings
warnings.filterwarnings('ignore')
import warnings
warnings.filterwarnings('ignore')

# Core imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import glob

# Import CLIP
from transformers import CLIPProcessor, CLIPModel

# Set random seeds
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("✓ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# ==================== CELL 3: CONFIGURATION ====================
class Config:
    # Paths - UPDATE THESE based on your Kaggle input
    DATA_PATH = "/kaggle/input/brisc2025/brisc2025"
    CLASSIFICATION_PATH = os.path.join(DATA_PATH, "classification_task")
    SEGMENTATION_PATH = os.path.join(DATA_PATH, "segmentation_task")
    SAVE_PATH = "/kaggle/working/"

    # Model parameters
    IMG_SIZE = 256
    BATCH_SIZE = 8  # Reduced for stability
    NUM_EPOCHS = 100
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4

    # Dataset parameters
    NUM_CLASSES = 4
    CLASS_NAMES = ['glioma', 'meningioma', 'no_tumor', 'pituitary']
    CLASS_MAPPING = {'glioma': 0, 'meningioma': 1, 'no_tumor': 2, 'pituitary': 3}

    # Loss weights
    WEIGHT_SEG = 3.0
    WEIGHT_CLS = 0.3

    # Training
    NUM_WORKERS = 0
    PIN_MEMORY = True
    EARLY_STOPPING_PATIENCE = 40

    # Text descriptions for CLIP
    TEXT_DESCRIPTIONS = {
        0: "MRI brain scan showing glioma tumor with irregular boundaries and infiltrative growth pattern in brain tissue",
        1: "MRI brain scan showing meningioma tumor with well-defined round borders arising from brain meninges membrane",
        2: "Normal healthy MRI brain scan without any tumor mass or abnormal tissue growth",
        3: "MRI brain scan showing pituitary adenoma tumor in the pituitary gland at skull base"
    }

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = Config()
print(f"\nConfiguration loaded")
print(f"Device: {config.DEVICE}")
print(f"Data path: {config.DATA_PATH}")

In [ ]:
# ==================== CELL 4: DATA LOADING ====================
def load_brisc2025_dataset(config):
    """Load BRISC2025 dataset"""
    print("Loading BRISC2025 dataset...")

    classification_data = []

    # Load from both train and test splits
    for split in ['train', 'test']:
        for class_name in config.CLASS_NAMES:
            class_path = os.path.join(config.CLASSIFICATION_PATH, split, class_name)

            if os.path.exists(class_path):
                for img_file in os.listdir(class_path):
                    if img_file.lower().endswith(('.jpg', '.png', '.jpeg')):
                        img_path = os.path.join(class_path, img_file)
                        classification_data.append({
                            'image_path': img_path,
                            'label': config.CLASS_MAPPING[class_name],
                            'class_name': class_name,
                            'split': split
                        })

    # Load segmentation masks if available
    segmentation_masks = {}
    for split in ['train', 'test']:
        mask_path = os.path.join(config.SEGMENTATION_PATH, split, 'masks')
        if os.path.exists(mask_path):
            for mask_file in os.listdir(mask_path):
                if mask_file.lower().endswith(('.png', '.jpg')):
                    mask_full_path = os.path.join(mask_path, mask_file)
                    # Try different naming conventions
                    img_name = mask_file.replace('_mask', '').replace('mask_', '')
                    segmentation_masks[img_name] = mask_full_path

    # Match masks to images
    for item in classification_data:
        img_filename = os.path.basename(item['image_path'])
        possible_names = [
            img_filename,
            img_filename.replace('.jpg', '.png'),
            'mask_' + img_filename,
            img_filename.replace('.jpg', '_mask.png')
        ]

        item['mask_path'] = None
        for name in possible_names:
            if name in segmentation_masks:
                item['mask_path'] = segmentation_masks[name]
                break

    df = pd.DataFrame(classification_data)

    print(f"\nDataset loaded!")
    print(f"Total images: {len(df)}")
    print(f"Images with masks: {df['mask_path'].notna().sum()}")
    print(f"\nClass distribution:")
    print(df['class_name'].value_counts())

    return df

# Load dataset
dataset_df = load_brisc2025_dataset(config)

# Split data
train_df = dataset_df[dataset_df['split'] == 'train'].reset_index(drop=True)
test_df = dataset_df[dataset_df['split'] == 'test'].reset_index(drop=True)

# Create validation split from train
if len(train_df) > 0:
    train_indices, val_indices = train_test_split(
        range(len(train_df)), 
        test_size=0.2, 
        stratify=train_df['label'].values,
        random_state=SEED
    )
    val_df = train_df.iloc[val_indices].reset_index(drop=True)
    train_df = train_df.iloc[train_indices].reset_index(drop=True)
else:
    # If no train folder, split test into train/val/test
    train_val_df, test_df = train_test_split(
        dataset_df, test_size=0.2, stratify=dataset_df['label'].values, random_state=SEED
    )
    train_df, val_df = train_test_split(
        train_val_df, test_size=0.25, stratify=train_val_df['label'].values, random_state=SEED
    )

print(f"\nFinal split:")
print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

In [ ]:
# ==================== TEXT-GUIDED ATTENTION MODULE ====================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class TextGuidedAttentionModule(nn.Module):
    def __init__(self, visual_channels, text_dim=512):
        super().__init__()
        self.visual_channels = visual_channels
        self.text_dim = text_dim

        self.text_spatial_proj = nn.Sequential(
            nn.Linear(text_dim, visual_channels),
            nn.ReLU(inplace=True),
            nn.Linear(visual_channels, visual_channels)
        )

        self.spatial_conv = nn.Sequential(
            nn.Conv2d(visual_channels, 1, kernel_size=1),
            nn.Sigmoid()
        )

        self.query_proj = nn.Conv2d(visual_channels, max(visual_channels // 8, 1), kernel_size=1)
        self.key_proj = nn.Linear(text_dim, max(visual_channels // 8, 1))
        self.value_proj = nn.Linear(text_dim, visual_channels)

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(visual_channels, max(visual_channels // 16, 1), kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(visual_channels // 16, 1), visual_channels, kernel_size=1),
            nn.Sigmoid()
        )

        self.beta = nn.Parameter(torch.tensor(0.5))

    def forward(self, visual_features, text_embedding):
        B, C, H, W = visual_features.shape

        text_spatial = self.text_spatial_proj(text_embedding)
        text_spatial = text_spatial.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, H, W)
        spatial_attn = self.spatial_conv(text_spatial)

        Q = self.query_proj(visual_features).view(B, -1, H * W).permute(0, 2, 1)
        K = self.key_proj(text_embedding).unsqueeze(1)
        V = self.value_proj(text_embedding).unsqueeze(1)

        attn_weights = torch.matmul(Q, K.transpose(1, 2)) / np.sqrt(max(C // 8, 1))
        attn_weights = F.softmax(attn_weights, dim=1)
        cross_attn_out = torch.matmul(attn_weights, V).permute(0, 2, 1).view(B, C, H, W)

        channel_attn = self.channel_attn(visual_features)
        out = visual_features * spatial_attn * channel_attn + self.beta * cross_attn_out

        return out

# ==================== ENCODER/DECODER BLOCKS ====================
class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upconv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv1 = nn.Conv2d(out_channels + skip_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, x, skip):
        x = self.upconv(x)
        x = torch.cat([x, skip], dim=1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x

# ==================== MAIN MODEL (MAX 256 CHANNELS) ====================
class TextGuidedUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes_seg=1, num_classes_cls=4, text_dim=512):
        super().__init__()
        
        # Encoder - max channel is now 256
        self.enc1 = EncoderBlock(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = EncoderBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = EncoderBlock(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = EncoderBlock(256, 256)  # Changed from 512 to 256
        self.pool4 = nn.MaxPool2d(2)
        
        # Bottleneck - changed from 512 to 256
        self.bottleneck = EncoderBlock(256, 256)
        
        # Text-Guided Attention
        self.tga1 = TextGuidedAttentionModule(64, text_dim)
        self.tga2 = TextGuidedAttentionModule(128, text_dim)
        self.tga3 = TextGuidedAttentionModule(256, text_dim)
        self.tga4 = TextGuidedAttentionModule(256, text_dim)  # Changed from 512 to 256
        
        # Decoder - adjusted for 256 max channels
        self.dec4 = DecoderBlock(in_channels=256, skip_channels=256, out_channels=128)  # Changed
        self.dec3 = DecoderBlock(in_channels=128, skip_channels=256, out_channels=64)   # Changed skip
        self.dec2 = DecoderBlock(in_channels=64, skip_channels=128, out_channels=64)
        self.dec1 = DecoderBlock(in_channels=64, skip_channels=64, out_channels=64)
        
        # Segmentation head
        self.seg_out = nn.Conv2d(64, num_classes_seg, kernel_size=1)
        
        # Classification head - adjusted for 256 bottleneck
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fusion_fc = nn.Linear(256 + text_dim, 256)  # Changed from 512+text_dim to 256+text_dim
        self.cls_fc1 = nn.Linear(256, 128)  # Changed from 512 to 256, output 128
        self.cls_fc2 = nn.Linear(128, num_classes_cls)  # Changed from 256 to 128
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x, text_embedding):
        # Encoder path
        e1 = self.enc1(x)      # [B, 64, 256, 256]
        p1 = self.pool1(e1)    # [B, 64, 128, 128]
        
        e2 = self.enc2(p1)     # [B, 128, 128, 128]
        p2 = self.pool2(e2)    # [B, 128, 64, 64]
        
        e3 = self.enc3(p2)     # [B, 256, 64, 64]
        p3 = self.pool3(e3)    # [B, 256, 32, 32]
        
        e4 = self.enc4(p3)     # [B, 256, 32, 32] - Changed from 512
        p4 = self.pool4(e4)    # [B, 256, 16, 16] - Changed from 512
        
        # Bottleneck
        b = self.bottleneck(p4)  # [B, 256, 16, 16] - Changed from 512
        
        # Text-guided attention on skip connections
        e1_tga = self.tga1(e1, text_embedding)  # [B, 64, 256, 256]
        e2_tga = self.tga2(e2, text_embedding)  # [B, 128, 128, 128]
        e3_tga = self.tga3(e3, text_embedding)  # [B, 256, 64, 64]
        e4_tga = self.tga4(e4, text_embedding)  # [B, 256, 32, 32] - Changed from 512
        
        # Decoder path
        d4 = self.dec4(b, e4_tga)        # [B, 128, 32, 32] - Changed
        d3 = self.dec3(d4, e3_tga)       # [B, 64, 64, 64] - Changed
        d2 = self.dec2(d3, e2_tga)       # [B, 64, 128, 128]
        d1 = self.dec1(d2, e1_tga)       # [B, 64, 256, 256]
        
        # Segmentation output
        seg_out = torch.sigmoid(self.seg_out(d1))  # [B, 1, 256, 256]
        
        # Classification output
        bottleneck_feat = self.global_pool(b).squeeze(-1).squeeze(-1)  # [B, 256] - Changed from 512
        fused_feat = torch.cat([bottleneck_feat, text_embedding], dim=1)  # [B, 768] - Changed
        fused_feat = F.relu(self.fusion_fc(fused_feat))  # [B, 256] - Changed
        cls_feat = F.relu(self.cls_fc1(self.dropout(fused_feat)))  # [B, 128] - Changed
        cls_out = self.cls_fc2(self.dropout(cls_feat))  # [B, 4]
        
        return seg_out, cls_out

print("✓ TextGuidedUNet model with max 256 channels")

# Example usage
if __name__ == "__main__":
    model = TextGuidedUNet(in_channels=3, num_classes_seg=1, num_classes_cls=4, text_dim=512)
    
    # Test forward pass
    batch_size = 2
    dummy_img = torch.randn(batch_size, 3, 256, 256)
    dummy_text = torch.randn(batch_size, 512)
    
    seg_out, cls_out = model(dummy_img, dummy_text)
    print(f"Segmentation output shape: {seg_out.shape}")  # [2, 1, 256, 256]
    print(f"Classification output shape: {cls_out.shape}")  # [2, 4]
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

In [ ]:
# ==================== CELL 8: DATASET WITH CACHED TEXT EMBEDDINGS ====================
class BRISC2025Dataset(Dataset):
    def __init__(self, dataframe, clip_model, clip_processor, transform=None, 
                 text_descriptions=None, precompute_text=True):
        self.df = dataframe
        self.transform = transform
        self.text_descriptions = text_descriptions or config.TEXT_DESCRIPTIONS
        
        # Precompute ALL text embeddings once (avoid CUDA in workers)
        if precompute_text:
            print("Precomputing text embeddings...")
            self.text_embeddings = {}
            with torch.no_grad():
                for label, text in self.text_descriptions.items():
                    text_inputs = clip_processor(text=[text], return_tensors="pt", 
                                                 padding=True, truncation=True, max_length=77)
                    text_inputs = {k: v.to(clip_model.device) for k, v in text_inputs.items()}
                    text_features = clip_model.get_text_features(**text_inputs)
                    self.text_embeddings[label] = text_features.squeeze(0).cpu()
            print(f"✓ Precomputed embeddings for {len(self.text_embeddings)} classes")
        else:
            self.text_embeddings = None
        
    def create_empty_mask(self, image_shape):
        """Create empty mask for no_tumor images"""
        return np.zeros((image_shape[0], image_shape[1]), dtype=np.float32)
    
    def create_synthetic_mask(self, image):
        """Create synthetic segmentation mask for tumor images"""
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image
        
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            mask = np.zeros_like(mask)
            cv2.drawContours(mask, [largest_contour], -1, 255, -1)
        
        return mask
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        image = cv2.imread(row['image_path'])
        if image is None:
            raise ValueError(f"Failed to load: {row['image_path']}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        label = int(row['label'])
        
        # Mask handling based on class
        if label == 2:  # no_tumor class
            mask = self.create_empty_mask(image.shape)
        elif pd.notna(row['mask_path']) and os.path.exists(row['mask_path']):
            mask = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
            mask = (mask > 127).astype(np.float32)
        else:
            mask = self.create_synthetic_mask(image)
            mask = (mask > 127).astype(np.float32)
        
        # Apply transforms
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        
        # Get precomputed text embedding (NO CUDA operations in worker)
        text_embedding = self.text_embeddings[label]
        
        mask = mask.unsqueeze(0)
        
        return image, mask, label, text_embedding

print("✓ Dataset class with precomputed text embeddings")


In [ ]:
# ==================== CELL 9: UPDATED LOSS FUNCTIONS ====================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = pred.contiguous().view(-1)
        target = target.contiguous().view(-1)
        
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        
        return 1 - dice

class CombinedLoss(nn.Module):
    def __init__(self, weight_seg=1.0, weight_cls=0.5):
        super().__init__()
        self.weight_seg = weight_seg
        self.weight_cls = weight_cls
        self.dice_loss = DiceLoss()
        self.bce_loss = nn.BCELoss()
        self.ce_loss = nn.CrossEntropyLoss()
    
    def forward(self, seg_pred, seg_target, cls_pred, cls_target):
        # Segmentation loss
        # For no_tumor class (label 2), the loss should penalize any predicted segmentation
        seg_loss = self.dice_loss(seg_pred, seg_target) + self.bce_loss(seg_pred, seg_target)
        
        # Classification loss
        cls_loss = self.ce_loss(cls_pred, cls_target)
        
        # Combined loss
        total_loss = self.weight_seg * seg_loss + self.weight_cls * cls_loss
        
        return total_loss, seg_loss, cls_loss

print("✓ Updated loss functions")


In [ ]:
# ==================== CELL 10: TRAINING FUNCTION ====================
def train_epoch(model, dataloader, optimizer, criterion, device, epoch):
    model.train()
    total_loss = 0
    total_seg_loss = 0
    total_cls_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc=f'Epoch {epoch+1} [Train]', leave=False)
    for images, masks, labels, text_embeddings in pbar:
        images = images.to(device)
        masks = masks.to(device)
        labels = labels.to(device)
        text_embeddings = text_embeddings.to(device)

        optimizer.zero_grad()

        seg_out, cls_out = model(images, text_embeddings)
        loss, seg_loss, cls_loss = criterion(seg_out, masks, cls_out, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_seg_loss += seg_loss.item()
        total_cls_loss += cls_loss.item()

        _, predicted = cls_out.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.1f}%'
        })

    return (total_loss / len(dataloader), 
            total_seg_loss / len(dataloader), 
            total_cls_loss / len(dataloader),
            100. * correct / total)

In [ ]:
# ==================== CELL 11: VALIDATION WITH DICE TRACKING ====================
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_seg_loss = 0
    total_cls_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_dice_scores = []
    
    with torch.no_grad():
        for images, masks, labels, text_embeddings in tqdm(dataloader, desc='Validating', leave=False):
            images = images.to(device)
            masks = masks.to(device)
            labels = labels.to(device)
            text_embeddings = text_embeddings.to(device)
            
            seg_out, cls_out = model(images, text_embeddings)
            loss, seg_loss, cls_loss = criterion(seg_out, masks, cls_out, labels)
            
            total_loss += loss.item()
            total_seg_loss += seg_loss.item()
            total_cls_loss += cls_loss.item()
            
            # Classification metrics
            _, predicted = cls_out.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Dice score calculation (only on tumor classes)
            labels_cpu = labels.cpu().numpy()
            tumor_mask = (labels_cpu == 0) | (labels_cpu == 1) | (labels_cpu == 3)
            
            if tumor_mask.any():
                seg_pred = (seg_out[tumor_mask] > 0.5).float()
                masks_tumor = masks[tumor_mask]
                
                intersection = (seg_pred * masks_tumor).sum(dim=(1,2,3))
                union = seg_pred.sum(dim=(1,2,3)) + masks_tumor.sum(dim=(1,2,3))
                dice = (2. * intersection + 1e-8) / (union + 1e-8)
                all_dice_scores.extend(dice.cpu().numpy())
    
    acc = 100. * correct / total
    mean_dice = np.mean(all_dice_scores) if len(all_dice_scores) > 0 else 0.0
    
    return (total_loss / len(dataloader), 
            total_seg_loss / len(dataloader), 
            total_cls_loss / len(dataloader),
            acc, mean_dice, all_preds, all_labels)

print("✓ Validation function with Dice tracking")


In [ ]:
# ==================== CELL 12: UPDATED TEST FUNCTION ====================
def test_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_seg_dice_tumor = []  # Only for tumor classes
    
    with torch.no_grad():
        for images, masks, labels, text_embeddings in tqdm(dataloader, desc='Testing'):
            images = images.to(device)
            masks = masks.to(device)
            labels_cpu = labels.cpu().numpy()
            text_embeddings = text_embeddings.to(device)
            
            seg_out, cls_out = model(images, text_embeddings)
            
            # Classification metrics
            _, predicted = cls_out.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels_cpu)
            
            # Segmentation metrics (only for tumor classes: 0=glioma, 1=meningioma, 3=pituitary)
            tumor_mask = (labels_cpu == 0) | (labels_cpu == 1) | (labels_cpu == 3)
            
            if tumor_mask.any():
                seg_pred = (seg_out[tumor_mask] > 0.5).float()
                masks_tumor = masks[tumor_mask]
                
                intersection = (seg_pred * masks_tumor).sum(dim=(1,2,3))
                union = seg_pred.sum(dim=(1,2,3)) + masks_tumor.sum(dim=(1,2,3))
                dice = (2. * intersection + 1e-8) / (union + 1e-8)
                all_seg_dice_tumor.extend(dice.cpu().numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    
    # Dice score only on tumor images
    mean_dice_tumor = np.mean(all_seg_dice_tumor) if len(all_seg_dice_tumor) > 0 else 0.0
    
    return accuracy, precision, recall, f1, mean_dice_tumor, all_preds, all_labels

print("✓ Updated test function")


In [ ]:
# ==================== CELL 13: UPDATED VISUALIZATION WITH DICE ====================
def plot_training_history(history):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Total Loss
    axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2)
    axes[0, 0].plot(history['val_loss'], label='Validation', linewidth=2)
    axes[0, 0].set_title('Total Loss', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Segmentation Loss
    axes[0, 1].plot(history['train_seg_loss'], label='Train', linewidth=2)
    axes[0, 1].plot(history['val_seg_loss'], label='Validation', linewidth=2)
    axes[0, 1].set_title('Segmentation Loss', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Classification Loss
    axes[0, 2].plot(history['train_cls_loss'], label='Train', linewidth=2)
    axes[0, 2].plot(history['val_cls_loss'], label='Validation', linewidth=2)
    axes[0, 2].set_title('Classification Loss', fontsize=14, fontweight='bold')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Loss')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Classification Accuracy
    axes[1, 0].plot(history['train_acc'], label='Train', linewidth=2)
    axes[1, 0].plot(history['val_acc'], label='Validation', linewidth=2)
    axes[1, 0].set_title('Classification Accuracy', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Accuracy (%)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Dice Score (NEW!)
    axes[1, 1].plot(history['val_dice'], label='Validation', linewidth=2, color='green')
    axes[1, 1].set_title('Segmentation Dice Score', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Dice Score')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].axhline(y=0.75, color='r', linestyle='--', alpha=0.5, label='Target (0.75)')
    
    # Combined Metric
    combined = [(history['val_acc'][i]/100 + history['val_dice'][i])/2 
                for i in range(len(history['val_acc']))]
    axes[1, 2].plot(combined, linewidth=2, color='purple')
    axes[1, 2].set_title('Combined Metric (Acc + Dice)', fontsize=14, fontweight='bold')
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_ylabel('Combined Score')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.SAVE_PATH, 'training_history.png'), dpi=300)
    plt.show()

# Keep other visualization functions same
def plot_confusion_matrix(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(config.SAVE_PATH, 'confusion_matrix.png'), dpi=300)
    plt.show()

def visualize_predictions(model, dataset, device, num_samples=4):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, mask, label, text_emb = dataset[idx]
            
            image_input = image.unsqueeze(0).to(device)
            text_input = text_emb.unsqueeze(0).to(device)
            
            seg_out, cls_out = model(image_input, text_input)
            _, pred_cls = cls_out.max(1)
            
            img_show = image.permute(1, 2, 0).cpu().numpy()
            img_show = img_show * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img_show = np.clip(img_show, 0, 1)
            
            axes[i, 0].imshow(img_show)
            axes[i, 0].set_title('Input Image')
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(mask.squeeze(), cmap='gray')
            axes[i, 1].set_title('Ground Truth Mask')
            axes[i, 1].axis('off')
            
            axes[i, 2].imshow(seg_out.squeeze().cpu().numpy(), cmap='gray')
            axes[i, 2].set_title('Predicted Mask')
            axes[i, 2].axis('off')
            
            true_class = config.CLASS_NAMES[label]
            pred_class = config.CLASS_NAMES[pred_cls.item()]
            color = 'green' if true_class == pred_class else 'red'
            
            axes[i, 3].text(0.5, 0.5, f'True: {true_class}\nPred: {pred_class}',
                           ha='center', va='center', fontsize=12,
                           bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
            axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.SAVE_PATH, 'predictions.png'), dpi=300)
    plt.show()

print("✓ Visualization functions with Dice plot")


In [ ]:
# ==================== CELL 14: MAIN TRAINING WITH DICE-BASED EARLY STOPPING ====================
def main():
    print("="*80)
    print("Text-Guided UNet Training on BRISC2025")
    print("="*80)
    
    # [Keep all the setup code from 1-4 same as before]
    # Load CLIP, transforms, datasets, model...
    
    print("\n[1/7] Loading CLIP model...")
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_model.eval()
    clip_model.to(config.DEVICE)
    print("✓ CLIP loaded")
    
    print("\n[2/7] Setting up transforms...")
    train_transform = A.Compose([
        A.Resize(config.IMG_SIZE, config.IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=20, p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.GaussNoise(p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    
    val_transform = A.Compose([
        A.Resize(config.IMG_SIZE, config.IMG_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    print("✓ Transforms ready")
    
    print("\n[3/7] Creating datasets...")
    train_dataset = BRISC2025Dataset(train_df, clip_model, clip_processor, 
                                     transform=train_transform, precompute_text=True)
    val_dataset = BRISC2025Dataset(val_df, clip_model, clip_processor, 
                                   transform=val_transform, precompute_text=True)
    test_dataset = BRISC2025Dataset(test_df, clip_model, clip_processor, 
                                    transform=val_transform, precompute_text=True)
    
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, 
                             shuffle=True, num_workers=config.NUM_WORKERS, 
                             pin_memory=config.PIN_MEMORY, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, 
                           shuffle=False, num_workers=config.NUM_WORKERS, 
                           pin_memory=config.PIN_MEMORY)
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, 
                            shuffle=False, num_workers=config.NUM_WORKERS, 
                            pin_memory=config.PIN_MEMORY)
    print("✓ Datasets created")
    
    print("\n[4/7] Initializing model...")
    model = TextGuidedUNet(in_channels=3, num_classes_seg=1, 
                          num_classes_cls=config.NUM_CLASSES, text_dim=512)
    model.to(config.DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"✓ Model initialized ({total_params:,} parameters)")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, 
                                 weight_decay=config.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.NUM_EPOCHS)
    criterion = CombinedLoss(weight_seg=config.WEIGHT_SEG, weight_cls=config.WEIGHT_CLS)
    
    # Training history - ADD DICE TRACKING
    history = {
        'train_loss': [], 'train_seg_loss': [], 'train_cls_loss': [], 'train_acc': [],
        'val_loss': [], 'val_seg_loss': [], 'val_cls_loss': [], 'val_acc': [], 'val_dice': []
    }
    
    best_combined_metric = 0  # Changed: track combined metric
    patience_counter = 0
    
    print("\n[5/7] Training...")
    print("="*80)
    
    for epoch in range(config.NUM_EPOCHS):
        train_loss, train_seg, train_cls, train_acc = train_epoch(
            model, train_loader, optimizer, criterion, config.DEVICE, epoch
        )
        
        val_loss, val_seg, val_cls, val_acc, val_dice, _, _ = validate(
            model, val_loader, criterion, config.DEVICE
        )
        
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['train_seg_loss'].append(train_seg)
        history['train_cls_loss'].append(train_cls)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_seg_loss'].append(val_seg)
        history['val_cls_loss'].append(val_cls)
        history['val_acc'].append(val_acc)
        history['val_dice'].append(val_dice)
        
        # Combined metric: Average of normalized accuracy and dice
        combined_metric = (val_acc / 100.0 + val_dice) / 2
        
        print(f"Epoch {epoch+1}/{config.NUM_EPOCHS}")
        print(f"  Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
        print(f"  Val:   Loss={val_loss:.4f}, Acc={val_acc:.2f}%, Dice={val_dice:.4f}")
        print(f"  Combined Metric: {combined_metric:.4f}")
        
        # Save best model based on combined metric
        if combined_metric > best_combined_metric:
            best_combined_metric = combined_metric
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(config.SAVE_PATH, 'best_model.pth'))
            print(f"  ✓ Best model saved! (Acc: {val_acc:.2f}%, Dice: {val_dice:.4f})")
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{config.EARLY_STOPPING_PATIENCE})")
        
        if patience_counter >= config.EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break
        print("-"*80)
    
    # Continue with testing and visualization...
    print("\n[6/7] Testing...")
    model.load_state_dict(torch.load(os.path.join(config.SAVE_PATH, 'best_model.pth')))
    
    test_acc, test_prec, test_rec, test_f1, test_dice, test_preds, test_labels = test_model(
        model, test_loader, config.DEVICE
    )
    
    print("\n" + "="*80)
    print("TEST RESULTS")
    print("="*80)
    print(f"Accuracy:  {test_acc*100:.2f}%")
    print(f"Precision: {test_prec:.4f}")
    print(f"Recall:    {test_rec:.4f}")
    print(f"F1-Score:  {test_f1:.4f}")
    print(f"Dice Score: {test_dice:.4f}")
    print("="*80)
    
    print("\n[7/7] Generating visualizations...")
    plot_training_history(history)
    plot_confusion_matrix(test_labels, test_preds, config.CLASS_NAMES)
    visualize_predictions(model, test_dataset, config.DEVICE, num_samples=4)
    
    results_df = pd.DataFrame({
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Dice'],
        'Value': [test_acc, test_prec, test_rec, test_f1, test_dice]
    })
    results_df.to_csv(os.path.join(config.SAVE_PATH, 'results.csv'), index=False)
    
    print("\n✓ Training complete! All outputs saved to /kaggle/working/")
    globals()['trained_model'] = model
    globals()['test_loader_global'] = test_loader
    globals()['train_loader_global'] = train_loader
    globals()['val_loader_global'] = val_loader

    
    return model, history


In [ ]:
# ==================== CELL 15: RUN TRAINING ====================
if __name__ == "__main__":
    model, history = main()

print("\n✓ All code cells ready to run!")

In [ ]:
# ==================== CELL 16: ADVANCED METRICS CALCULATION ====================
import cv2
from scipy.ndimage import distance_transform_edt
from sklearn.metrics import jaccard_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm

def calculate_dice_score(pred, target):
    """Calculate Dice score"""
    pred = pred.flatten()
    target = target.flatten()
    intersection = (pred * target).sum()
    dice = (2. * intersection + 1e-8) / (pred.sum() + target.sum() + 1e-8)
    return dice

def calculate_iou(pred, target):
    """Calculate IoU (Intersection over Union)"""
    pred = pred.flatten()
    target = target.flatten()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    iou = (intersection + 1e-8) / (union + 1e-8)
    return iou

def calculate_hausdorff_distance_95(pred, target):
    """Calculate 95th percentile Hausdorff Distance"""
    pred_binary = (pred > 0.5).astype(np.uint8)
    target_binary = (target > 0.5).astype(np.uint8)

    if pred_binary.sum() == 0 and target_binary.sum() == 0:
        return 0.0
    if pred_binary.sum() == 0 or target_binary.sum() == 0:
        return 999.0

    pred_boundary = pred_binary - cv2.erode(pred_binary, np.ones((3,3), np.uint8))
    target_boundary = target_binary - cv2.erode(target_binary, np.ones((3,3), np.uint8))

    pred_dt = distance_transform_edt(1 - pred_boundary)
    target_dt = distance_transform_edt(1 - target_boundary)

    pred_distances = target_dt[pred_boundary > 0]
    target_distances = pred_dt[target_boundary > 0]

    if len(pred_distances) == 0 or len(target_distances) == 0:
        return 0.0

    hd95_pred = np.percentile(pred_distances, 95)
    hd95_target = np.percentile(target_distances, 95)
    hd95 = max(hd95_pred, hd95_target)

    return hd95


def calculate_weighted_iou(class_metrics, class_counts):
    """
    Calculate weighted IoU (weighted by number of samples per class)
    
    Args:
        class_metrics: dict with per-class IoU scores
        class_counts: dict with number of samples per class
    
    Returns:
        weighted_iou: float
    """
    total_samples = sum(class_counts.values())
    weighted_sum = 0.0
    
    for cls_idx in class_counts.keys():
        weight = class_counts[cls_idx] / total_samples
        weighted_sum += weight * class_metrics[cls_idx]['iou']
    
    return weighted_sum


def compute_comprehensive_metrics(model, dataloader, device, config):
    """Compute all metrics including per-class breakdown and weighted IoU"""
    model.eval()
    
    all_dice_scores = []
    all_iou_scores = []
    all_hd95_scores = []
    
    class_dice = {0: [], 1: [], 3: []}
    class_iou = {0: [], 1: [], 3: []}
    class_hd95 = {0: [], 1: [], 3: []}
    class_counts = {0: 0, 1: 0, 3: 0}  # NEW: Count samples per class
    
    all_results = []
    
    with torch.no_grad():
        for batch_idx, (images, masks, labels, text_embeddings) in enumerate(tqdm(dataloader, desc='Computing metrics')):
            images = images.to(device)
            masks = masks.to(device)
            labels_cpu = labels.cpu().numpy()
            text_embeddings = text_embeddings.to(device)
            
            seg_out, cls_out = model(images, text_embeddings)
            
            for i in range(images.shape[0]):
                label = labels_cpu[i]
                
                if label == 2:
                    continue
                
                pred_mask = (seg_out[i].squeeze().cpu().numpy() > 0.5).astype(np.float32)
                true_mask = masks[i].squeeze().cpu().numpy().astype(np.float32)
                
                dice = calculate_dice_score(pred_mask, true_mask)
                iou = calculate_iou(pred_mask, true_mask)
                hd95 = calculate_hausdorff_distance_95(pred_mask, true_mask)
                
                all_dice_scores.append(dice)
                all_iou_scores.append(iou)
                all_hd95_scores.append(hd95)
                
                class_dice[label].append(dice)
                class_iou[label].append(iou)
                class_hd95[label].append(hd95)
                class_counts[label] += 1  # NEW: Count samples
                
                all_results.append({
                    'batch_idx': batch_idx,
                    'sample_idx': i,
                    'image': images[i].cpu(),
                    'pred_mask': pred_mask,
                    'true_mask': true_mask,
                    'label': label,
                    'dice': dice,
                    'iou': iou,
                    'hd95': hd95,
                    'text_emb': text_embeddings[i].cpu()
                })
    
    avg_dice = np.mean(all_dice_scores)
    avg_iou = np.mean(all_iou_scores)
    avg_hd95 = np.mean(all_hd95_scores)
    
    class_metrics = {}
    for cls in [0, 1, 3]:
        class_metrics[cls] = {
            'dice': np.mean(class_dice[cls]) if len(class_dice[cls]) > 0 else 0,
            'iou': np.mean(class_iou[cls]) if len(class_iou[cls]) > 0 else 0,
            'hd95': np.mean(class_hd95[cls]) if len(class_hd95[cls]) > 0 else 0
        }
    
    # NEW: Calculate weighted IoU
    weighted_iou = calculate_weighted_iou(class_metrics, class_counts)
    
    return {
        'avg_dice': avg_dice,
        'avg_iou': avg_iou,
        'weighted_iou': weighted_iou,  # NEW
        'avg_hd95': avg_hd95,
        'class_metrics': class_metrics,
        'class_counts': class_counts,  # NEW
        'all_results': all_results
    }

print("✓ Advanced metrics functions defined")

In [ ]:
# ==================== CELL 17: COMPUTE METRICS (USES GLOBAL VARIABLES) ====================
# Access the global variables
model_to_use = trained_model
loader_to_use = test_loader_global

print("\nComputing comprehensive metrics...")
metrics = compute_comprehensive_metrics(model_to_use, loader_to_use, config.DEVICE, config)

print("\n" + "="*80)
print("COMPREHENSIVE SEGMENTATION METRICS")
print("="*80)
print(f"\n1. Average DICE Score:        {metrics['avg_dice']:.4f}")
print(f"2. Average mIOU (unweighted):  {metrics['avg_iou']:.4f}")
print(f"3. Weighted mIOU:              {metrics['weighted_iou']:.4f}")
print(f"4. Average HD95:               {metrics['avg_hd95']:.2f} pixels")

print("\n" + "-"*80)
print("Per-Class mIOU:")
print("-"*80)
for cls_idx in [0, 1, 3]:
    cls_name = config.CLASS_NAMES[cls_idx]
    iou = metrics['class_metrics'][cls_idx]['iou']
    print(f"  {cls_name:12s}: {iou:.4f}")

print("\n" + "-"*80)
print("Per-Class DICE:")
print("-"*80)
for cls_idx in [0, 1, 3]:
    cls_name = config.CLASS_NAMES[cls_idx]
    dice = metrics['class_metrics'][cls_idx]['dice']
    print(f"  {cls_name:12s}: {dice:.4f}")

print("\n" + "-"*80)
print("Per-Class HD95:")
print("-"*80)
for cls_idx in [0, 1, 3]:
    cls_name = config.CLASS_NAMES[cls_idx]
    hd95 = metrics['class_metrics'][cls_idx]['hd95']
    print(f"  {cls_name:12s}: {hd95:.2f} pixels")
print("="*80)

print("\n" + "-"*80)
print("Class Distribution (counts):")
print("-"*80)
for cls_idx in [0, 1, 3]:
    cls_name = config.CLASS_NAMES[cls_idx]
    count = metrics['class_counts'][cls_idx]
    percentage = 100 * count / sum(metrics['class_counts'].values())
    print(f"  {cls_name:12s}: {count:4d} samples ({percentage:.1f}%)")

print("\n" + "-"*80)
print("Per-Class mIOU:")
print("-"*80)
for cls_idx in [0, 1, 3]:
    cls_name = config.CLASS_NAMES[cls_idx]
    iou = metrics['class_metrics'][cls_idx]['iou']
    count = metrics['class_counts'][cls_idx]
    weight = count / sum(metrics['class_counts'].values())
    print(f"  {cls_name:12s}: {iou:.4f} (weight: {weight:.3f})")
print("="*80)

# Save metrics
metrics_df = pd.DataFrame({
    'Metric': ['Average DICE', 'Average mIOU', 'Weighted mIOU', 'Average HD95'],
    'Value': [metrics['avg_dice'], metrics['avg_iou'], metrics['weighted_iou'], metrics['avg_hd95']]
})

class_metrics_df = pd.DataFrame({
    'Class': [config.CLASS_NAMES[i] for i in [0, 1, 3]],
    'Count': [metrics['class_counts'][i] for i in [0, 1, 3]],
    'Weight': [metrics['class_counts'][i]/sum(metrics['class_counts'].values()) for i in [0, 1, 3]],
    'DICE': [metrics['class_metrics'][i]['dice'] for i in [0, 1, 3]],
    'mIOU': [metrics['class_metrics'][i]['iou'] for i in [0, 1, 3]],
    'HD95': [metrics['class_metrics'][i]['hd95'] for i in [0, 1, 3]]
})

metrics_df.to_csv(os.path.join(config.SAVE_PATH, 'overall_metrics.csv'), index=False)
class_metrics_df.to_csv(os.path.join(config.SAVE_PATH, 'per_class_metrics.csv'), index=False)
print("\n✓ Metrics saved to CSV files")


In [ ]:
# ==================== CELL 18: EXTRACT BEST/WORST SAMPLES ====================
all_results = metrics['all_results']
all_results_sorted = sorted(all_results, key=lambda x: x['dice'])

worst_sample = all_results_sorted[0]
best_sample = all_results_sorted[-1]

print(f"\nBest sample - DICE: {best_sample['dice']:.4f}, Class: {config.CLASS_NAMES[best_sample['label']]}")
print(f"Worst sample - DICE: {worst_sample['dice']:.4f}, Class: {config.CLASS_NAMES[worst_sample['label']]}")


In [ ]:
# ==================== CELL 19: FEATURE VISUALIZATION ====================
def extract_intermediate_features(model, image, text_embedding):
    """Extract features from all encoder and decoder blocks"""
    model.eval()
    features = {}

    with torch.no_grad():
        x = image.unsqueeze(0)
        text_emb = text_embedding.unsqueeze(0)

        # Encoder
        e1 = model.enc1(x)
        features['enc1'] = e1
        p1 = model.pool1(e1)

        e2 = model.enc2(p1)
        features['enc2'] = e2
        p2 = model.pool2(e2)

        e3 = model.enc3(p2)
        features['enc3'] = e3
        p3 = model.pool3(e3)

        e4 = model.enc4(p3)
        features['enc4'] = e4
        p4 = model.pool4(e4)

        # Bottleneck
        b = model.bottleneck(p4)
        features['bottleneck'] = b

        # Text-guided attention
        e1_tga = model.tga1(e1, text_emb)
        features['enc1_tga'] = e1_tga
        e2_tga = model.tga2(e2, text_emb)
        features['enc2_tga'] = e2_tga
        e3_tga = model.tga3(e3, text_emb)
        features['enc3_tga'] = e3_tga
        e4_tga = model.tga4(e4, text_emb)
        features['enc4_tga'] = e4_tga

        # Decoder
        d4 = model.dec4(b, e4_tga)
        features['dec4'] = d4
        d3 = model.dec3(d4, e3_tga)
        features['dec3'] = d3
        d2 = model.dec2(d3, e2_tga)
        features['dec2'] = d2
        d1 = model.dec1(d2, e1_tga)
        features['dec1'] = d1

        # Final
        seg_out = torch.sigmoid(model.seg_out(d1))
        features['final_seg'] = seg_out

    return features

def visualize_features(features, sample_name, save_path):
    """Visualize all features"""
    fig = plt.figure(figsize=(20, 12))

    feature_names = ['enc1', 'enc2', 'enc3', 'enc4', 'bottleneck', 
                     'dec4', 'dec3', 'dec2', 'dec1', 'final_seg']

    for idx, feat_name in enumerate(feature_names):
        ax = plt.subplot(2, 5, idx + 1)

        feat = features[feat_name].squeeze().cpu().numpy()

        if len(feat.shape) == 3:
            feat = feat.mean(axis=0)

        im = ax.imshow(feat, cmap='viridis')
        ax.set_title(f'{feat_name}\nShape: {feat.shape}', fontsize=10, fontweight='bold')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.suptitle(f'Feature Maps - {sample_name}', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f'features_{sample_name}.png'), dpi=300, bbox_inches='tight')
    plt.show()

# Visualize best
print("\nGenerating feature visualizations for BEST sample...")
best_features = extract_intermediate_features(
    model_to_use, 
    best_sample['image'].to(config.DEVICE), 
    best_sample['text_emb'].to(config.DEVICE)
)
visualize_features(best_features, 'best_sample', config.SAVE_PATH)

# Visualize worst
print("Generating feature visualizations for WORST sample...")
worst_features = extract_intermediate_features(
    model_to_use,
    worst_sample['image'].to(config.DEVICE),
    worst_sample['text_emb'].to(config.DEVICE)
)
visualize_features(worst_features, 'worst_sample', config.SAVE_PATH)

print("✓ Feature visualizations saved")

In [ ]:
# ==================== CELL 20: OVERLAY VISUALIZATION ====================
def create_overlay_image(true_mask, pred_mask):
    """Create color-coded overlay"""
    h, w = true_mask.shape
    overlay = np.zeros((h, w, 3), dtype=np.uint8)

    true_mask_binary = (true_mask > 0.5).astype(bool)
    pred_mask_binary = (pred_mask > 0.5).astype(bool)

    # True Positive - GREEN
    tp = true_mask_binary & pred_mask_binary
    overlay[tp] = [0, 255, 0]

    # False Positive - RED
    fp = pred_mask_binary & ~true_mask_binary
    overlay[fp] = [255, 0, 0]

    # False Negative - BLUE
    fn = true_mask_binary & ~pred_mask_binary
    overlay[fn] = [0, 0, 255]

    return overlay

In [ ]:
# ==================== CELL 21: GRID OF 5 SAMPLES ====================
np.random.seed(42)
random_indices = np.random.choice(len(all_results), size=min(5, len(all_results)), replace=False)
selected_samples = [all_results[i] for i in random_indices]

fig, axes = plt.subplots(5, 4, figsize=(16, 20))

for row_idx, sample in enumerate(selected_samples):
    img = sample['image'].permute(1, 2, 0).cpu().numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)

    true_mask = sample['true_mask']
    pred_mask = sample['pred_mask']
    label = sample['label']
    dice = sample['dice']

    overlay = create_overlay_image(true_mask, pred_mask)

    # Original
    axes[row_idx, 0].imshow(img)
    axes[row_idx, 0].set_title(f'Original\n{config.CLASS_NAMES[label]}', fontsize=10, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Ground Truth
    axes[row_idx, 1].imshow(true_mask, cmap='gray')
    axes[row_idx, 1].set_title('Ground Truth', fontsize=10, fontweight='bold')
    axes[row_idx, 1].axis('off')

    # Predicted
    axes[row_idx, 2].imshow(pred_mask, cmap='gray')
    axes[row_idx, 2].set_title(f'Predicted\nDICE: {dice:.3f}', fontsize=10, fontweight='bold')
    axes[row_idx, 2].axis('off')

    # Overlay
    axes[row_idx, 3].imshow(overlay)
    axes[row_idx, 3].set_title('Overlay', fontsize=10, fontweight='bold')
    axes[row_idx, 3].axis('off')

# Legend
legend_elements = [
    mpatches.Patch(color='green', label='Correct (TP)'),
    mpatches.Patch(color='red', label='False Positive'),
    mpatches.Patch(color='blue', label='False Negative')
]
fig.legend(handles=legend_elements, loc='upper center', ncol=3, 
          fontsize=12, frameon=True, bbox_to_anchor=(0.5, 0.99))

plt.suptitle('Segmentation Results: Original | Ground Truth | Predicted | Overlay', 
            fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.985])
plt.savefig(os.path.join(config.SAVE_PATH, 'segmentation_grid_5samples.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Grid visualization saved")

In [ ]:
# ==================== CELL 22: BEST/WORST COMPARISON ====================
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

samples_to_show = [
    ('BEST Sample', best_sample),
    ('WORST Sample', worst_sample)
]

for row_idx, (title, sample) in enumerate(samples_to_show):
    img = sample['image'].permute(1, 2, 0).cpu().numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)

    true_mask = sample['true_mask']
    pred_mask = sample['pred_mask']
    label = sample['label']
    dice = sample['dice']
    iou = sample['iou']
    hd95 = sample['hd95']

    overlay = create_overlay_image(true_mask, pred_mask)

    # Original
    axes[row_idx, 0].imshow(img)
    axes[row_idx, 0].set_title(f'{title}\n{config.CLASS_NAMES[label]}', fontsize=11, fontweight='bold')
    axes[row_idx, 0].axis('off')

    # Ground Truth
    axes[row_idx, 1].imshow(true_mask, cmap='gray')
    axes[row_idx, 1].set_title('Ground Truth', fontsize=11, fontweight='bold')
    axes[row_idx, 1].axis('off')

    # Predicted
    axes[row_idx, 2].imshow(pred_mask, cmap='gray')
    axes[row_idx, 2].set_title(f'Predicted\nDICE: {dice:.3f} | IoU: {iou:.3f}', 
                               fontsize=11, fontweight='bold')
    axes[row_idx, 2].axis('off')

    # Overlay
    axes[row_idx, 3].imshow(overlay)
    axes[row_idx, 3].set_title(f'Overlay\nHD95: {hd95:.2f}px', fontsize=11, fontweight='bold')
    axes[row_idx, 3].axis('off')

# Legend
legend_elements = [
    mpatches.Patch(color='green', label='True Positive'),
    mpatches.Patch(color='red', label='False Positive'),
    mpatches.Patch(color='blue', label='False Negative')
]
fig.legend(handles=legend_elements, loc='upper center', ncol=3,
          fontsize=11, frameon=True, bbox_to_anchor=(0.5, 0.98))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(config.SAVE_PATH, 'best_worst_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Best/Worst comparison saved")

print("\n" + "="*80)
print("ALL VISUALIZATIONS AND METRICS COMPLETE!")
print("="*80)
print("\nGenerated files:")
print("  1. overall_metrics.csv")
print("  2. per_class_metrics.csv")
print("  3. features_best_sample.png")
print("  4. features_worst_sample.png")
print("  5. segmentation_grid_5samples.png")
print("  6. best_worst_comparison.png")
print("="*80)
